In [81]:
import torch
from torch.utils.data import DataLoader, Dataset
import pydicom as dicom
import pandas as pd
import numpy as np
import unet

import skimage
import scipy

import matplotlib.pyplot as plt

In [82]:
def crop_to_mask(image, mask, padding=20):
    # Find nonzero coordinates
    coords = np.argwhere(mask)

    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0)

    # Add padding
    y_min = max(y_min - padding, 0)
    x_min = max(x_min - padding, 0)

    y_max = min(y_max + padding, image.shape[0])
    x_max = min(x_max + padding, image.shape[1])

    # Crop
    cropped_image = image[y_min:y_max, x_min:x_max]
    cropped_mask  = mask[y_min:y_max, x_min:x_max]

    return cropped_image, cropped_mask

def preprocess(image: np.array, mask: np.array, enlarging_factor: int = 2) -> tuple:
    IMAGE_TARGET_SHAPE = (572, 572)
    MASK_TARGET_SHAPE = (388, 388)
    
    ## Preprocess the image
    # application of median filtering
    median_filtered_image = scipy.ndimage.median_filter(image, size=3)

    # application of clahe
    clahe_image = skimage.exposure.equalize_adapthist(median_filtered_image)
    
    # application of unsharp
    preproc_image = skimage.filters.unsharp_mask(clahe_image)

    ## Preprocess the mask
    preproc_image, preproc_mask = crop_to_mask(preproc_image, mask)

    # downscales both the image and the mask
    preproc_image = skimage.transform.resize(preproc_image, IMAGE_TARGET_SHAPE) 
    preproc_mask = skimage.transform.resize(preproc_mask, MASK_TARGET_SHAPE)
    
    return preproc_image, preproc_mask

In [83]:
class CBIS_Dataset(Dataset):
    def __init__(self, dataset_filepath, train, transform=None):
        self.lesions_df = pd.read_csv(dataset_filepath)
        self.transform = transform
        
        # keeping only MLO and masses
        self.lesions_df = self.lesions_df[(self.lesions_df["image view"] == "MLO") & (self.lesions_df["kind"] == "Mass")]

        # based on parameter, keep only training or test instances
        if train:
            self.lesions_df = self.lesions_df[self.lesions_df["training or test"] == "training"]
        else:
            self.lesions_df = self.lesions_df[self.lesions_df["training or test"] == "Test"]
        
        self.images = []
        self.masks = []

        # load and preprocess the lesions
        for index, row in self.lesions_df.iterrows():
            image = dicom.dcmread(row["fullimage filepath"]).pixel_array
            mask = dicom.dcmread(row["roi filepath"]).pixel_array

            preproc_image, preproc_mask = preprocess(image, mask)

            # cast the image to tensor
            tensor_image = torch.as_tensor(preproc_image.astype(np.float32)).unsqueeze(0)

            # Create one-hot encoded channels
            channel_0 = (preproc_mask == 0).astype(np.float32)
            channel_1 = (preproc_mask != 0).astype(np.float32)
            stacked_array = np.stack([channel_0, channel_1], axis=-1)
            
            tensor_mask = torch.from_numpy(np.transpose(stacked_array, (2,0,1)))

            self.images.append(tensor_image)
            self.masks.append(tensor_mask)

        self.n_elements = len(self.images)
    
    def __len__(self):
        return self.n_elements

    def __getitem__(self, idx):
        image, mask = self.images[idx], self.masks[idx]

        if self.transform:
            image = self.transform(image)

        return image, mask

In [84]:
learning_rate = 1e-3
batch_size = 50
epochs = 3

# loading data
train_data = CBIS_Dataset(dataset_filepath=f"{EXPORT_FILEPATH}/lesions.csv", train=True)
test_data = CBIS_Dataset(dataset_filepath=f"{EXPORT_FILEPATH}/lesions.csv", train=False)

train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

# defyning loss function
loss_fn = unet.dice_loss

model = unet.UNet(n_class=2)

# defyning optimizer
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=learning_rate
)

In [ ]:
# training loop
for t in range(epochs):
    print(f"Epoch {t+1}\n------------")
    unet.train_loop(train_dataloader, model, loss_fn, optimizer, batch_size)
    unet.test_loop(test_dataloader, model, loss_fn)

Epoch 1
------------
